# 02 - Versionando um agente com MLflow

Caso prático: um **atendente virtual** de uma loja. Aqui versionamos três coisas separadas:

1. **Código do agente**: o arquivo `agente/agente_atendimento.py`, versionado no git.
2. **Prompt**: o texto de instrução, registrado no Prompt Registry do MLflow, com versões.
3. **Configuração do modelo**: o nome do modelo, guardado junto com a versão do agente.

Cada registro do agente é uma **versão do modelo** (`agente-atendimento`) gerada por um **run** num experimento separado (`agente-atendimento`). Os experimentos do notebook anterior não são alterados.

Fluxo:

```
prompt v1 ─┐                              ┌─ run "agente-prompt-v1" → agente versão 1
           ├─ agente_atendimento.py ──────┤
prompt v2 ─┘                              └─ run "agente-prompt-v2" → agente versão 2
                                                       │
                              mlflow models serve ─────┴─ POST /invocations (formato de chat da OpenAI)
```

- **Models from code**: o MLflow registra o arquivo `.py` e carrega a classe `ChatModel` a partir dele, sem serializar objetos Python.
- **`model_config`**: os valores que mudam entre versões (a instrução e o modelo) entram como configuração do registro.
- **Chamada**: o servidor expõe o agente pelo endpoint `/invocations`, com entrada e saída no formato de mensagens de chat da OpenAI.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto. O banco do MLflow é o mesmo `mlflow.db` do notebook anterior, mas o experimento é outro.

In [1]:
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import mlflow
import requests
import yaml
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from mlflow import MlflowClient

from IPython.display import Markdown, display
from IPython.display import Image, Code

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENTO = "agente-atendimento"
NOME_AGENTE = "agente-atendimento"
NOME_PROMPT = "agente-atendimento-prompt"
MODELO = os.getenv("OPENAI_MODEL", "gpt-5-mini")

ARQUIVO_YAML = Path("prompts/agente_atendimento.yaml")
ARQUIVO_CODIGO = Path("agente/agente_atendimento.py")
ARQUIVO_YAML.parent.mkdir(exist_ok=True)

experimento = mlflow.set_experiment(EXPERIMENTO)
print(f"Experimento '{experimento.name}' (id {experimento.experiment_id})")

2026/11/16 22:10:48 INFO mlflow.tracking.fluent: Experiment with name 'agente-atendimento' does not exist. Creating a new experiment.


Experimento 'agente-atendimento' (id 1)


## 2. Prompt do agente (versão 1)

O prompt usa a variável `{{ nome_loja }}`, preenchida na hora de gerar a instrução do agente. A mensagem de commit descreve a versão.

In [2]:
YAML_V1 = """
commit: "Versão inicial do atendente"
system: |
  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial e objetiva.
  Se não tiver certeza da resposta, diga que não sabe e sugira falar com um atendente humano.
"""

ARQUIVO_YAML.write_text(YAML_V1, encoding="utf-8")
print(ARQUIVO_YAML.read_text(encoding="utf-8"))


commit: "Versão inicial do atendente"
system: |
  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial e objetiva.
  Se não tiver certeza da resposta, diga que não sabe e sugira falar com um atendente humano.



`registrar_prompt` lê o YAML e registra as mensagens no Prompt Registry, como no notebook anterior.

In [3]:
def registrar_prompt(caminho: Path):
    dados = yaml.safe_load(caminho.read_text(encoding="utf-8"))
    mensagens = [{"role": "system", "content": dados["system"]}]
    return mlflow.genai.register_prompt(
        name=NOME_PROMPT,
        template=mensagens,
        commit_message=dados["commit"],
    )


versao_prompt_1 = registrar_prompt(ARQUIVO_YAML)
print(f"Prompt registrado: {versao_prompt_1.name} versão {versao_prompt_1.version}")

Prompt registrado: agente-atendimento-prompt versão 1


## 3. Código do agente

O agente é a classe `AgenteAtendimento`, em `agente/agente_atendimento.py`. Ela não tem prompt fixo: a instrução chega por `model_config` e é usada em toda chamada. Veja o código que será registrado:

In [4]:
display(Code(
    filename=str(ARQUIVO_CODIGO),
    language="python"
))

"""Agente de atendimento versionado com MLflow (models from code).

O código fica neste arquivo, versionado no git. A instrução (o prompt já formatado)
e o modelo chegam pelo `model_config` no momento do registro, então cada versão do
agente guarda exatamente o prompt que foi usado.
"""

import os

from mlflow.models import set_model
from mlflow.pyfunc import ChatModel
from mlflow.types.llm import ChatChoice, ChatCompletionResponse, ChatMessage, ChatParams
from openai import OpenAI


class AgenteAtendimento(ChatModel):
    def load_context(self, context):
        config = context.model_config or {}
        self.instrucao = config["instrucao"]
        self.modelo = config.get("modelo", "gpt-5-mini")
        self.cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def predict(self, context, messages: list[ChatMessage], params: ChatParams) -> ChatCompletionResponse:
        historico = [{"role": "system", "content": self.instrucao}]
        historico += [{"role": m.role, "content": m.content} for m in messages]

        resposta = self.cliente.chat.completions.create(model=self.modelo, messages=historico)
        texto = resposta.choices[0].message.content

        return ChatCompletionResponse(
            choices=[ChatChoice(message=ChatMessage(role="assistant", content=texto))]
        )


set_model(AgenteAtendimento())

## 4. Registrando o agente

`instrucao_da_versao` busca a versão do prompt e preenche a variável `nome_loja`. `registrar_agente` abre um run no experimento e registra:

- **params**: a URI do prompt, a versão do prompt, o modelo e o caminho do código;
- **tags** e **artefato** `instrucao.txt`: a descrição e o texto exato da instrução;
- **modelo**: o código do agente com o `model_config`, registrado como nova versão de `agente-atendimento`.

In [5]:
def instrucao_da_versao(versao: int) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{NOME_PROMPT}/{versao}")
    mensagens = prompt.format(nome_loja="Loja Exemplo")
    return mensagens[0]["content"]


def registrar_agente(versao_prompt: int, descricao: str):
    instrucao = instrucao_da_versao(versao_prompt)

    with mlflow.start_run(run_name=f"agente-prompt-v{versao_prompt}"):
        mlflow.log_params({
            "prompt_uri": f"prompts:/{NOME_PROMPT}/{versao_prompt}",
            "prompt_versao": versao_prompt,
            "modelo": MODELO,
            "codigo": str(ARQUIVO_CODIGO),
        })
        mlflow.set_tag("descricao", descricao)
        mlflow.log_text(instrucao, "instrucao.txt")

        info = mlflow.pyfunc.log_model(
            name="agente",
            python_model=str(ARQUIVO_CODIGO.resolve()),
            model_config={"instrucao": instrucao, "modelo": MODELO},
            registered_model_name=NOME_AGENTE,
            pip_requirements=["openai"],
        )

    print(f"Agente registrado: {NOME_AGENTE} versão {info.registered_model_version}")
    return info


info_agente_1 = registrar_agente(1, "Prompt inicial")

/home/leonardo/Documentos/github/loop_engineering_study/notebooks/03-MLFlow/agente/agente_atendimento.py:35: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  set_model(AgenteAtendimento())
2026/11/16 22:10:48 WARNING mlflow.pyfunc: Default values for temperature, n and stream in ChatParams will be removed in the next release. Specify them in the input example explicitly if needed.
2026/11/16 22:10:48 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/11/16 22:10:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/tmp/tmp24_f3qzq/model/agente_atendimento.py:35: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  set_model(AgenteAtendimento())
202

Agente registrado: agente-atendimento versão 1


Successfully registered model 'agente-atendimento'.
Created version '1' of model 'agente-atendimento'.


## 5. Alterando o prompt (versão 2)

A mudança é pequena: limitar a resposta a três frases e orientar o cliente em pedidos de reembolso. Editamos o YAML, mostramos o diff, registramos o prompt v2 e, em seguida, um novo agente que usa essa versão.

In [6]:
YAML_V2 = """
commit: "Respostas de até 3 frases e orientação sobre reembolso"
system: |
  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial e objetiva,
  em no máximo 3 frases.
  Em pedidos de reembolso, oriente o cliente a consultar a política de devoluções.
  Se não tiver certeza da resposta, diga que não sabe e sugira falar com um atendente humano.
"""

ARQUIVO_YAML.write_text(YAML_V2, encoding="utf-8")

import difflib
print("".join(difflib.unified_diff(YAML_V1.splitlines(True), YAML_V2.splitlines(True), "v1", "v2")))

versao_prompt_2 = registrar_prompt(ARQUIVO_YAML)
print(f"Prompt registrado: {versao_prompt_2.name} versão {versao_prompt_2.version}")

info_agente_2 = registrar_agente(2, "Prompt v2: até 3 frases e orientação sobre reembolso")

/home/leonardo/Documentos/github/loop_engineering_study/notebooks/03-MLFlow/agente/agente_atendimento.py:35: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  set_model(AgenteAtendimento())
2026/11/16 22:10:54 WARNING mlflow.pyfunc: Default values for temperature, n and stream in ChatParams will be removed in the next release. Specify them in the input example explicitly if needed.
2026/11/16 22:10:54 INFO mlflow.pyfunc: Predicting on input example to validate output


--- v1
+++ v2
@@ -1,5 +1,7 @@
 
-commit: "Versão inicial do atendente"
+commit: "Respostas de até 3 frases e orientação sobre reembolso"
 system: |
-  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial e objetiva.
+  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial e objetiva,
+  em no máximo 3 frases.
+  Em pedidos de reembolso, oriente o cliente a consultar a política de devoluções.
   Se não tiver certeza da resposta, diga que não sabe e sugira falar com um atendente humano.

Prompt registrado: agente-atendimento-prompt versão 2


2026/11/16 22:10:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/tmp/tmpnmp6aeb4/model/agente_atendimento.py:35: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  set_model(AgenteAtendimento())
2026/11/16 22:10:59 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Agente registrado: agente-atendimento versão 2


Registered model 'agente-atendimento' already exists. Creating a new version of this model...
Created version '2' of model 'agente-atendimento'.


## 6. Comparando as versões

Os runs ficam no experimento `agente-atendimento`, e cada um guarda o prompt que gerou a versão do agente.

In [7]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO], order_by=["start_time ASC"])
colunas = ["run_id", "tags.mlflow.runName", "params.prompt_versao", "params.modelo", "tags.descricao"]
display(runs[colunas])

for versao in MlflowClient().search_model_versions(f"name='{NOME_AGENTE}'"):
    print(f"Agente versão {versao.version} → run {versao.run_id}")

,run_id,tags.mlflow.runName,params.prompt_versao,params.modelo,tags.descricao
0,8075fba21e9a4798a0140b41521d87e4,agente-prompt-v1,1,gpt-5-mini,Prompt inicial
1,f56db9c9d565474b803935b829ad8658,agente-prompt-v2,2,gpt-5-mini,Prompt v2: até 3 frases e orientação sobre ree...


Agente versão 2 → run f56db9c9d565474b803935b829ad8658
Agente versão 1 → run 8075fba21e9a4798a0140b41521d87e4


## 7. Servindo o agente

A função `servir` sobe o agente com `mlflow models serve` numa porta local, espera o servidor responder em `/health` e o encerra ao sair do bloco `with`. O servidor usa o `.env` carregado acima, porque herda as variáveis de ambiente.

O cliente envia a conversa no formato de mensagens de chat da OpenAI e recebe a resposta no formato `choices[0].message`.

Observação: o endpoint é `/invocations`, não `/v1/chat/completions`. Por isso o SDK oficial da OpenAI não aponta direto para o servidor; o que é compatível é o formato de entrada e de saída.

In [8]:
@contextmanager
def servir(uri: str, porta: int):
    ambiente = {**os.environ, "MLFLOW_TRACKING_URI": "sqlite:///mlflow.db"}
    processo = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "models", "serve", "-m", uri, "-p", str(porta), "--env-manager", "local"],
        cwd=Path.cwd(),
        env=ambiente,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    base = f"http://127.0.0.1:{porta}"
    try:
        inicio = time.time()
        while True:
            try:
                if requests.get(f"{base}/health", timeout=2).status_code == 200:
                    break
            except requests.ConnectionError:
                pass
            if processo.poll() is not None or time.time() - inicio > 180:
                raise RuntimeError(f"O servidor do MLflow não subiu para {uri}.")
            time.sleep(2)
        yield base
    finally:
        processo.terminate()
        processo.wait(timeout=30)


def conversar(base: str, pergunta: str) -> str:
    corpo = {"messages": [{"role": "user", "content": pergunta}]}
    resposta = requests.post(f"{base}/invocations", json=corpo, timeout=120)
    resposta.raise_for_status()
    return resposta.json()["choices"][0]["message"]["content"]

## 8. Chamando as duas versões

Subimos a versão 1 e a versão 2 uma de cada vez e fazemos as mesmas perguntas em cada uma. A diferença nas respostas vem só do prompt.

In [9]:
PERGUNTAS = [
    "Posso devolver um produto depois de 10 dias?",
    "Quero um reembolso por Pix, como funciona?",
]

for versao, porta in [(1, 5001), (2, 5002)]:
    with servir(f"models:/{NOME_AGENTE}/{versao}", porta) as base:
        linhas = [f"## Agente versão {versao}", ""]
        for pergunta in PERGUNTAS:
            linhas.append(f"**Pergunta:** {pergunta}")
            linhas.append("")
            linhas.append(f"**Resposta:** {conversar(base, pergunta)}")
            linhas.append("")
        display(Markdown("\n".join(linhas)))

## Agente versão 1

**Pergunta:** Posso devolver um produto depois de 10 dias?

**Resposta:** Depende.

- Se a compra foi feita online (entrega em domicílio), pela lei brasileira (direito de arrependimento) você tem 7 dias, a partir do recebimento, para devolver sem justificar — portanto, após 10 dias normalmente não é possível usar esse direito.  
- Se houve defeito no produto ou se ainda estiver na garantia, há regras diferentes e pode ser possível troca/assistência mesmo depois de 10 dias.  
- Também pode haver uma política própria da Loja Exemplo com prazo maior; preciso checar seu pedido.

Quer que eu verifique para você? Me informe o número do pedido, a data de recebimento e o motivo da devolução (arrependimento, defeito, outro). Se preferir falar com um atendente humano, eu encaminho.

**Pergunta:** Quero um reembolso por Pix, como funciona?

**Resposta:** Posso ajudar — seguem as informações gerais e o que preciso para iniciar o reembolso por Pix.

Como funciona (resumo)
- Elegibilidade: o reembolso depende da nossa política de devolução/estorno (prazo para solicitar, estado do produto etc.).  
- Dados necessários: número do pedido, CPF/CNPJ do comprador e a chave Pix para a devolução (CPF, e‑mail, telefone ou chave aleatória). Se preferir, podemos devolver para conta bancária (banco, agência, conta e tipo).  
- Prazo: quando a loja envia o Pix, o crédito normalmente entra em segundos/minutos. Porém o nosso processo interno de análise e liberação do estorno pode levar até alguns dias úteis (costuma variar entre 1–7 dias úteis, dependendo do caso).  
- Confirmação: você recebe uma confirmação por e‑mail ou mensagem com o comprovante do estorno.

O que eu preciso para iniciar
- Número do pedido  
- CPF/CNPJ do titular  
- Chave Pix para recebimento (ou dados bancários, se preferir)  
- Motivo do reembolso (opcional, mas ajuda na análise)

Observações importantes
- Preferimos fazer o reembolso para a mesma chave/conta usada no pagamento, quando possível.  
- Se houver divergência de titularidade da chave Pix, podemos solicitar documentação adicional por segurança.

Quer que eu abra a solicitação agora? Se sim, me informe o número do pedido e a chave Pix (ou diga se prefere conta bancária). Se preferir falar com um atendente humano, eu encaminho.


## Agente versão 2

**Pergunta:** Posso devolver um produto depois de 10 dias?

**Resposta:** Olá — não sei ao certo, pois isso depende da nossa política de devoluções e do estado do produto.  
Por favor, consulte a política de devoluções da Loja Exemplo para prazos e condições.  
Se preferir, fale com um atendente humano para verificar seu caso específico.

**Pergunta:** Quero um reembolso por Pix, como funciona?

**Resposta:** Por favor, consulte nossa política de devoluções para ver requisitos e prazos para reembolso via Pix. Geralmente é necessário abrir a solicitação pelo suporte e informar sua chave Pix para o estorno, mas não sei os detalhes do seu caso; fale com um atendente humano para orientação personalizada.


## 9. Observações

- **Reexecução**: cada execução cria novos runs, novas versões de prompt e novas versões do agente. Para recomeçar, apague `mlflow.db` e a pasta `prompts/` (mantenha o `.gitkeep`, se existir).
- **Interface**: com o servidor do MLflow da seção `03-MLFlow` rodando (`mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000`), o experimento `agente-atendimento` aparece com os runs, e o modelo `agente-atendimento` com as versões.
- **Aviso do MLflow**: na versão 3.18, a classe `ChatModel` aparece como depreciada, e o MLflow sugere `ResponsesAgent`. Mantivemos `ChatModel` porque ele expõe o formato de chat da OpenAI usado aqui. Os avisos de depreciação aparecem na execução.

## 10. Servindo e chamando o agente pelo terminal

Esta seção faz manualmente o que a função `servir` faz no notebook: sobe o agente num servidor e chama pelo `curl`. Use dois terminais.

### Terminal 1: subir o servidor

```bash
cd notebooks/03-MLFlow
set -a && source ../../.env && set +a
MLFLOW_TRACKING_URI=sqlite:///mlflow.db uv run --project ../.. mlflow models serve -m models:/agente-atendimento/2 -p 5002 --env-manager local
```

- `cd notebooks/03-MLFlow`: o caminho `sqlite:///mlflow.db` é relativo à pasta atual, então o terminal precisa estar nela.
- `set -a && source ../../.env && set +a`: exporta a `OPENAI_API_KEY` do `.env` para o processo. O MLflow não lê o `.env` sozinho.
- `MLFLOW_TRACKING_URI=sqlite:///mlflow.db`: indica onde está o registro do modelo. Sem isso, o MLflow procura em `./mlruns`, não encontra o agente e falha.
- `uv run --project ../.. mlflow models serve`: roda o comando no ambiente do projeto.
- `-m models:/agente-atendimento/2`: a versão do agente a servir. Troque `/2` por `/1` para servir a versão 1.
- `-p 5002`: a porta. Ela precisa estar livre.
- `--env-manager local`: usa o ambiente atual, sem criar outro.

Quando aparecer `Uvicorn running on http://127.0.0.1:5002`, o servidor está pronto. Para parar, pressione `Ctrl+C` nesse terminal.

### Terminal 2: chamando com curl

Primeiro, confira se o servidor está no ar:

```bash
curl http://127.0.0.1:5002/health
```

Depois, envie uma pergunta no formato de mensagens de chat da OpenAI:

```bash
curl -s http://127.0.0.1:5002/invocations \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Posso devolver um produto depois de 10 dias?"}]}'
```

A resposta é JSON. O texto do atendente fica em `choices[0].message.content`. Para ver só esse texto, sem instalar nada extra:

```bash
curl -s http://127.0.0.1:5002/invocations \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Quero um reembolso por Pix, como funciona?"}]}' \
  | python3 -c "import sys, json; print(json.load(sys.stdin)['choices'][0]['message']['content'])"
```

Algumas observações:

- O endpoint é `/invocations`, não `/v1/chat/completions`. Clientes que esperam o caminho da OpenAI precisam apontar para `/invocations`.
- O agente não guarda histórico. Para continuar uma conversa, envie no array `messages` as mensagens anteriores, alternando `user` e `assistant`, junto com a nova pergunta.
- Para testar a outra versão, rode o servidor de novo com `-m models:/agente-atendimento/1`, ou troque a porta para não conflitar.